# MCP Server Distillation — End-to-End Tutorial

**Goal:** Automatically generate high-quality tool-use training data from a frontier model, then fine-tune a small language model to become an expert at using your custom MCP server tools.

## The Problem

You've built a custom MCP server (APIs, tools, data sources) and connected it to a small language model (e.g., Qwen3-4B). But the small model struggles:
- Picks the wrong tool when several look similar
- Extracts parameters incorrectly from natural language
- Fails multi-step tasks that require chaining tools together
- Doesn't understand the relationships between your tools

## The Solution: MCP Server Distillation

Instead of manually writing training examples, we use a **frontier model** (e.g., GPT-5.2) to automatically generate expert-quality training data for your specific MCP server.

The pipeline has three key phases:

### Phase 1: Explore
The frontier model connects to your MCP server and **actively calls every tool** — discovering real data entities, learning how tools relate to each other, identifying which tools are easily confused, and finding edge cases.

### Phase 2: Generate
Using the exploration findings, a teacher LLM synthesizes realistic user questions that require multi-tool reasoning. The frontier model then **solves each question** by actually calling your MCP tools, producing expert-quality tool-use trajectories.

### Phase 3: Format & Train
The structured tool traces are converted into the **function-calling conversation format** used for supervised fine-tuning. Then we train the student model with LoRA GRPO.

```
  ┌───────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
  │  Expert   │   │ Question │   │ Quality  │   │ Expert   │   │ Format   │   │   GRPO   │
  │Exploration│──▶│Synthesis │──▶│ Filter   │──▶│Trajectory│──▶│ Training │──▶│ Training │
  │           │   │          │   │          │   │          │   │   Data   │   │          │
  └───────────┘   └──────────┘   └──────────┘   └──────────┘   └──────────┘   └──────────┘
   Frontier model  Teacher LLM   Teacher LLM    Frontier model  Structured     LoRA GRPO
   calls MCP tools generates Qs  scores quality  solves via MCP  messages       fine-tune
```

## What You Need

1. **Your MCP server** running (this tutorial uses a demo e-commerce server with 15 tools)
2. **Langflow** with a frontier model agent connected to your MCP server
3. **An API key** for the teacher LLM (used for question generation + quality scoring)
4. **GPU(s)** for GRPO training (Step 6)

---
## Step 0: Setup

Install dependencies and configure credentials.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install sdg_hub nest_asyncio pandas fastmcp training_hub

In [ ]:
from pathlib import Path
import json
import os
import sys

import nest_asyncio
import pandas as pd

nest_asyncio.apply()

NOTEBOOK_DIR = Path.cwd()
print(f"Working directory: {NOTEBOOK_DIR}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
# Set these via environment variables or edit directly
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-...")
TEACHER_MODEL = os.environ.get("TEACHER_MODEL", "openai/gpt-5.2")

# Langflow flow with FRONTIER MODEL (GPT-5.2) + MCP server connection
# This is NOT the student model — the frontier model generates expert trajectories
LANGFLOW_URL = os.environ.get(
    "LANGFLOW_URL",
    "http://localhost:3000/api/v1/run/<your-flow-id>",
)
LANGFLOW_API_KEY = os.environ.get("LANGFLOW_API_KEY", None)

# Student model for training
STUDENT_MODEL = os.environ.get("STUDENT_MODEL", "Qwen/Qwen3-4B")

# Output directories
OUTPUT_DIR = NOTEBOOK_DIR / "generated_data"
CHECKPOINT_DIR = NOTEBOOK_DIR / "checkpoints"
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

print(f"Teacher model:  {TEACHER_MODEL}")
print(f"Langflow URL:   {LANGFLOW_URL}")
print(f"Student model:  {STUDENT_MODEL}")
print(f"Output dir:     {OUTPUT_DIR}")

---
## Step 1: Understand the Demo MCP Server

Before generating training data, let's understand what we're working with. This tutorial uses the **ShopInsights Analytics Platform** — a demo e-commerce MCP server with 15 tools located in `demo_server/`.

### Start the demo server

In a separate terminal:
```bash
cd demo_server/
pip install fastmcp
python server.py
```

The server will be available at `http://localhost:8008`.

### Ambiguity Clusters

The tools are deliberately designed with **ambiguity clusters** — groups of tools that look similar but serve different purposes. This is exactly the kind of challenge that small models struggle with:

| Cluster | Tools | Why it's hard for small models |
|---|---|---|
| Product Discovery | `search_products`, `browse_catalog`, `get_trending_products`, `get_product_details` | Which search/browse tool is right for a given query? |
| Sales & Revenue | `get_sales_data`, `get_revenue_report`, `get_store_overview` | Per-product units vs. aggregate revenue vs. quick snapshot |
| Customer Analytics | `get_customer_segments`, `get_customer_profile`, `get_abandoned_carts` | Individual lookup vs. aggregate analysis vs. behavioral data |
| Multi-step | `analyze_product_performance`, `compare_products`, `forecast_demand`, `get_inventory_status`, `create_promotion` | Require chaining 2-4 tools in the correct order |

In [ ]:
# Inspect the demo server's data (works without the server running)
sys.path.insert(0, str(NOTEBOOK_DIR / "demo_server"))
from data import create_data_store

store = create_data_store()

print(f"Products:        {len(store.products)}")
print(f"Customers:       {len(store.customers)}")
print(f"Orders:          {len(store.orders)}")
print(f"Inventory rows:  {len(store.inventory)}")
print(f"Abandoned carts: {len(store.abandoned_carts)}")
print(f"Promotions:      {len(store.promotions)}")

print("\nCategory hierarchy:")
for cat in sorted({p["category"] for p in store.products}):
    count = sum(1 for p in store.products if p["category"] == cat)
    print(f"  {cat} ({count} products)")

In [ ]:
# List all 15 tools from the server
from server import mcp

tools = await mcp.list_tools()

print(f"{'#':<3} {'Tool Name':<32} {'Params':>6}  Description")
print("\u2500" * 100)
for i, t in enumerate(tools, 1):
    mt = t.to_mcp_tool()
    n_params = len(mt.inputSchema.get("properties", {}))
    desc = (mt.description or "").split("\n")[0][:50]
    print(f"{i:<3} {mt.name:<32} {n_params:>6}  {desc}")

In [ ]:
# Quick smoke test: call a few tools directly
from server import get_store_overview, search_products

print("=== Store Overview ===")
overview = get_store_overview()
print(f"  Revenue: ${overview['total_revenue']:,.2f}")
print(f"  Orders:  {overview['total_orders']}")
print(f"  AOV:     ${overview['average_order_value']:,.2f}")
print("  Top products:")
for p in overview["top_5_products"]:
    print(f"    {p['product_id']}: {p['name']} (${p['revenue']:,.2f})")

print("\n=== Search: 'wireless' ===")
results = search_products(query="wireless", limit=3)
for p in results["products"]:
    print(f"  {p['id']}: {p['name']} \u2014 ${p['price']:.2f}")

---
## Step 2: Create Input Dataset & Configure Langflow

The distillation pipeline needs a simple input: a DataFrame with your MCP server's tool schemas, name, and description.

| Column | Type | Description |
|---|---|---|
| `tool_list` | `list[dict]` | Each tool's `name`, `description`, and `inputSchema` (JSON Schema) |
| `mcp_server_name` | `str` | Human-readable server name |
| `mcp_server_description` | `str` | What the server does |

### Langflow Setup

Before running the pipeline, you need a Langflow flow with:
1. A **frontier model** (e.g., GPT-5.2) as the LLM
2. Your **MCP server** connected as a tool provider
3. An **agent** component that chains the LLM with the MCP tools

The Langflow agent acts as our "expert" — it will explore the MCP server and generate gold-standard tool-use demonstrations that we later use to train the small student model.

In [ ]:
# Build the input dataset from server tool schemas
tool_list = []
for t in tools:
    mt = t.to_mcp_tool()
    tool_list.append(
        {
            "name": mt.name,
            "description": mt.description or "",
            "inputSchema": mt.inputSchema,
        }
    )

server_name = mcp.name or "ShopInsights Analytics Platform"
server_description = (
    "E-commerce analytics platform for an online retailer. "
    "Provides product search, sales analytics, customer insights, "
    "demand forecasting, and promotional management. "
    "Features 15 tools organized across product discovery, sales & revenue, "
    "customer analytics, and multi-step analytical workflows."
)

input_df = pd.DataFrame(
    {
        "tool_list": [tool_list],
        "mcp_server_name": [server_name],
        "mcp_server_description": [server_description],
    }
)

print(f"Input: {len(input_df)} row(s), {len(tool_list)} tools")
print(f"Server: {server_name}")

---
## Step 3: Run the SDG Hub Distillation Pipeline

The pipeline is defined as a YAML flow with **23 blocks** across **6 stages**:

| Stage | What happens | Why it matters |
|---|---|---|
| 1. **Expert Exploration** | Frontier model connects to your MCP server and calls every tool | Discovers real data, tool relationships, and edge cases |
| 2. **Diversity** | Multiplies rows and samples different tool subsets | Ensures training data covers diverse tool combinations |
| 3. **Question Synthesis** | Teacher LLM generates realistic questions using exploration findings | Questions reference real entities, not hypothetical ones |
| 4. **Question Quality Filter** | Scores questions on difficulty, realism, uniqueness | Removes trivial or poorly-formed questions |
| 5. **Expert Trajectories** | Frontier model solves each question via actual MCP tool calls | Produces gold-standard demonstrations of correct tool use |
| 6. **Response Quality Filter** | Scores trajectories on completeness and conciseness | Removes incomplete or verbose responses |

### Expected data flow (starting from 1 row with 15 tools)

```
1 row  →  Exploration (adds server_understanding)
       →  ×10 multiplier = 10 rows
       →  Sample 2-tool subsets per row
       →  Generate questions (10 candidate questions)
       →  Quality filter → ~6-8 questions survive
       →  Expert trajectories (frontier model solves each)
       →  Response quality filter → final training examples
```

In [ ]:
from sdg_hub import Flow, FlowRegistry

# Auto-discover all built-in flows
FlowRegistry.discover_flows()

# Load the MCP distillation flow by its deterministic ID
flow_id = "new-night-835"
flow_path = FlowRegistry.get_flow_path(flow_id)
flow = Flow.from_yaml(flow_path)

print(f"Flow: {flow.metadata.name}")
print(f"Version: {flow.metadata.version}")
print(f"Tags: {flow.metadata.tags}")
print(f"Blocks: {len(flow.blocks)}")
print(f"Agent blocks: {flow._detect_agent_blocks()}")
print(f"\nRecommended model: {flow.get_model_recommendations()}")

In [ ]:
# Configure the teacher model (LLM API calls for question gen + quality scoring)
flow.set_model_config(
    model=TEACHER_MODEL,
    api_key=OPENAI_API_KEY,
)
print(f"Teacher model configured: {TEACHER_MODEL}")

# Configure the frontier model agent (Langflow + MCP server)
agent_kwargs = {
    "agent_framework": "langflow",
    "agent_url": LANGFLOW_URL,
}
if LANGFLOW_API_KEY:
    agent_kwargs["agent_api_key"] = LANGFLOW_API_KEY

flow.set_agent_config(**agent_kwargs)
print(f"Agent configured for: {flow._detect_agent_blocks()}")
print(f"Langflow URL: {LANGFLOW_URL}")

# Longer timeout for exploration (it calls many tools sequentially)
flow.set_agent_config(timeout=300, blocks=["explore_server"])
print("Exploration timeout set to 300s")

In [ ]:
# Run the full distillation pipeline
# This takes several minutes — the frontier model will explore your MCP server,
# generate questions, and solve each one with actual tool calls.
print(f"Input: {len(input_df)} row(s), {len(input_df['tool_list'][0])} tools")
print("Running pipeline...")
print("-" * 60)

result = flow.generate(
    input_df,
    runtime_params={
        "multiply_tool_rows": {"num_samples": 10},
        "sample_tools": {"num_samples": 2},
    },
    checkpoint_dir=str(CHECKPOINT_DIR),
)

print("-" * 60)
print(f"Pipeline complete! Generated {len(result)} training examples")

In [ ]:
# Convert result to pandas for analysis
if hasattr(result, "to_pandas"):
    df = result.to_pandas()
else:
    df = result

print(f"Generated {len(df)} training examples")
print(f"Columns: {list(df.columns)}")

---
## Step 3b: Exploration Deep Dive

The exploration is the most important step. Unlike approaches that only read tool schemas, our pipeline has the frontier model **actually call the tools** and observe their behavior.

The exploration produces a structured "server understanding" document that captures:
- **Data entities** — real product IDs, customer names, category hierarchies, price ranges
- **Tool relationships** — which tool outputs feed into which tool inputs
- **Ambiguity clusters** — which tools overlap in functionality and when to use each one
- **Edge cases** — error conditions, empty results, parameter constraints

In [ ]:
# Inspect the exploration findings
if "server_understanding" in df.columns:
    exploration = df["server_understanding"].iloc[0]
    print("=" * 80)
    print("SERVER UNDERSTANDING (from frontier model exploration)")
    print("=" * 80)
    print(exploration[:3000])
    if len(exploration) > 3000:
        print(f"\n... ({len(exploration) - 3000} more characters)")
else:
    print("server_understanding column not found")
    print(f"Available columns: {list(df.columns)}")

---
## Step 3c: Results Analysis

Each surviving row contains:
- A **realistic question** grounded in exploration findings
- The **target tools** that should be used to answer it
- An **expert trajectory** — the frontier model's actual tool calls and final answer
- **Quality scores** from the teacher LLM

In [ ]:
# Summary of generated examples
key_cols = [
    c
    for c in [
        "question",
        "target_tools",
        "question_quality_rating",
        "completeness_rating",
        "conciseness_rating",
    ]
    if c in df.columns
]
if key_cols:
    display(df[key_cols])

In [ ]:
# Quality distribution
for col in ["question_quality_rating", "completeness_rating", "conciseness_rating"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().to_string())

---
## Step 4: Format Training Data

The pipeline outputs an `extract_agent_text_tool_trace` column — the full structured tool call trace from the Langflow agent. Each trace contains:

- **Input** — the user's question
- **Tool calls** — each tool the expert called, with exact arguments and output
- **Output** — the final synthesized answer

We convert these traces into the **function-calling conversation format** used for SFT:

```
[system]    "Here are the tools you can use: [tool schemas...]"
[user]      "Find the top trending laptop and check inventory"
[assistant] → function_call: get_trending_products(metric="revenue", ...)
[function]  ← {"trending": [{"id": "PROD-0006", ...}]}
[assistant] → function_call: get_inventory_status(product_ids=["PROD-0006"])
[function]  ← {"inventory": [...], "low_stock_count": 0}
[assistant] "Here's what I found: the AeroBook Gaming X17 is trending..."
```

In [ ]:
# Inspect the raw tool traces
if "extract_agent_text_tool_trace" in df.columns:
    trace = df["extract_agent_text_tool_trace"].iloc[0]
    print(f"First example has {len(trace)} trace steps:")
    for i, step in enumerate(trace):
        role = step.get("role")
        if role == "assistant" and step.get("tool_calls"):
            for tc in step["tool_calls"]:
                name = tc.get("function", {}).get("name", "")
                print(f"  [{i}] tool_call -> {name}()")
        elif role == "tool":
            print(f"  [{i}] tool result [{step.get('name', '')}]")
        elif role == "assistant":
            content = str(step.get("content", ""))[:80]
            print(f"  [{i}] assistant: {content}")
        else:
            print(f"  [{i}] {role or 'unknown'}")
else:
    print("Tool trace column not found. Available columns:")
    print(list(df.columns))

In [ ]:
# Format all rows into structured conversations using SDG Hub's formatter
from sdg_hub.core.utils.message_formatter import tool_trace_to_messages

formatted_messages = []
skipped = 0

for idx, row in df.iterrows():
    trace = row["extract_agent_text_tool_trace"]
    tools_schema = row["tool_list"]
    msgs = tool_trace_to_messages(trace, tools_schema)
    if msgs:
        formatted_messages.append(json.dumps(msgs))
    else:
        skipped += 1

df["messages"] = formatted_messages + [None] * skipped
df_valid = df[df["messages"].notna()].copy()

print(f"Formatted {len(df_valid)} rows into structured conversations")
if skipped:
    print(f"Skipped {skipped} rows with empty/malformed traces")

for idx, row in df_valid.head(5).iterrows():
    msgs = json.loads(row["messages"])
    roles = [m["role"] for m in msgs]
    n_tool_calls = sum(1 for m in msgs if "function_call" in m)
    print(f"  Row {idx}: {len(msgs)} messages, {n_tool_calls} tool call(s)")

In [ ]:
# Inspect a formatted example in detail
sample_msgs = json.loads(df_valid["messages"].iloc[0])

print(f"Example conversation ({len(sample_msgs)} messages):\n")
for i, msg in enumerate(sample_msgs):
    role = msg["role"]

    if role == "system":
        content = msg["content"]
        tools_start = content.find("[")
        tools_end = content.rfind("]") + 1
        if tools_start >= 0 and tools_end > tools_start:
            declared = json.loads(content[tools_start:tools_end])
            print(f"[{i}] system \u2014 declares {len(declared)} tools")
        else:
            print(f"[{i}] system \u2014 {content[:80]}...")

    elif "function_call" in msg:
        fc = msg["function_call"]
        print(f"\n[{i}] assistant \u2192 calls {fc['name']}()")
        print(f"    arguments: {fc['arguments'][:100]}")

    elif role == "function":
        content = msg["content"]
        print(f"[{i}] function ({msg['name']}) \u2192 {len(content)} chars")

    elif role == "user":
        print(f"\n[{i}] user")
        print(f"    {msg['content'][:120]}")

    elif role == "assistant":
        print(f"\n[{i}] assistant (final answer)")
        print(f"    {msg['content'][:200]}...")

In [ ]:
# Export training data
export_cols = [
    "messages",
    "question",
    "target_tools",
    "question_quality_rating",
    "completeness_rating",
]
df_export = df_valid[[c for c in export_cols if c in df_valid.columns]].copy()

# Save as JSONL (primary training format)
jsonl_path = OUTPUT_DIR / "training_data.jsonl"
with open(jsonl_path, "w") as f:
    for _, row in df_export.iterrows():
        f.write(json.dumps({"messages": json.loads(row["messages"])}) + "\n")

# Also save as Parquet for analysis
parquet_path = OUTPUT_DIR / "distillation_output.parquet"
df_export.to_parquet(parquet_path, index=False)

print(f"Exported {len(df_export)} training examples")
print(f"  JSONL:   {jsonl_path}")
print(f"  Parquet: {parquet_path}")

---
## Step 5: Train with LoRA GRPO

Now we train the student model (e.g., Qwen3-4B) using **Group Relative Policy Optimization (GRPO)**. This reinforcement learning method:

1. Generates multiple completions for each prompt
2. Scores them using a reward model (tool-use correctness)
3. Updates the model to prefer better completions relative to worse ones

Combined with **LoRA** (Low-Rank Adaptation), we get efficient fine-tuning that only trains a small adapter — keeping the base model frozen.

### Backends

| Backend | GPUs | Best for |
|---------|------|----------|
| `art` | 1 | Prototyping, small datasets |
| `verl` | 2-8 | Production training, large datasets |

In [ ]:
from training_hub import lora_grpo

# Count training examples
with open(jsonl_path) as f:
    n_examples = sum(1 for _ in f)

print("=" * 60)
print("MCP Distillation \u2014 GRPO Training")
print("=" * 60)
print(f"  Student model:    {STUDENT_MODEL}")
print(f"  Training data:    {jsonl_path} ({n_examples} examples)")
print(f"  Backend:          art (single-GPU)")
print(f"  LoRA rank:        32")
print(f"  LoRA alpha:       64")
print(f"  Iterations:       15")
print(f"  Group size:       8")
print("=" * 60)

In [ ]:
# Run GRPO training (requires GPU)
train_result = lora_grpo(
    model_path=STUDENT_MODEL,
    data_path=str(jsonl_path),
    ckpt_output_dir=str(CHECKPOINT_DIR),
    backend="art",
    lora_r=32,
    lora_alpha=64,
    num_iterations=15,
    group_size=8,
    prompt_batch_size=100,
    learning_rate=1e-5,
)

print("\nTraining complete!")
if isinstance(train_result, dict):
    for key, value in train_result.items():
        print(f"  {key}: {value}")
else:
    print(f"  Result: {train_result}")

print(f"\nCheckpoints saved to: {CHECKPOINT_DIR}")

---
## Step 6: Evaluate the Fine-Tuned Model

After training, evaluate the student model on tool-use tasks to verify it has learned to:
1. Select the correct tool from ambiguous options
2. Extract parameters accurately from natural language
3. Chain multiple tools for complex queries
4. Produce coherent final answers from tool outputs

In [ ]:
# Define evaluation queries that test different capabilities
eval_queries = [
    # Single tool, correct selection from ambiguous cluster
    "What are the current top trending products by revenue?",
    "Give me a quick overview of overall store performance.",
    "Look up customer alice@example.com",
    
    # Multi-tool chaining
    "Find the top-selling laptop and check if we have enough inventory for the next 30 days.",
    "Compare the two most expensive phones and analyze which performs better.",
    
    # Complex parameter extraction
    "Search for wireless accessories under $50 sorted by rating.",
    "Get monthly revenue breakdown by region from January to June 2025.",
]

print(f"Prepared {len(eval_queries)} evaluation queries")
for i, q in enumerate(eval_queries, 1):
    print(f"  {i}. {q}")

In [ ]:
# Evaluate the trained model
# Load the fine-tuned model checkpoint and run inference against the MCP server.
# Compare tool selection accuracy vs. the frontier model's gold trajectories.
#
# Metrics to track:
#   - Tool selection accuracy: did it pick the right tool(s)?
#   - Argument accuracy: did it extract correct parameters?
#   - Task completion: did it solve the full query?
#   - Response quality: is the final answer coherent and complete?

print("Evaluation requires loading the trained checkpoint and running inference.")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print("\nTo evaluate with vLLM:")
print(f"  from vllm import LLM")
print(f"  model = LLM('{STUDENT_MODEL}', lora_path='{CHECKPOINT_DIR}/final')")
print("\nThen run each eval query and compare tool calls against gold trajectories.")

---
## Scaling Up

The pipeline is designed to scale. Increase volume and question complexity via runtime parameter overrides:

| Block | Parameter | Default | What it controls |
|---|---|---|---|
| `multiply_tool_rows` | `num_samples` | 10 | How many question candidates to generate |
| `sample_tools` | `num_samples` | 2 | How many tools each question must involve |

### Example: 50 examples with 3-tool questions

```python
result = flow.generate(
    input_df,
    runtime_params={
        "multiply_tool_rows": {"num_samples": 50},
        "sample_tools": {"num_samples": 3},
    },
    checkpoint_dir="./checkpoints",
)
```

### Multiple MCP servers

If you have multiple MCP servers, create a multi-row input DataFrame:

```python
df = pd.DataFrame({
    "tool_list": [server1_tools, server2_tools, server3_tools],
    "mcp_server_name": ["Payments API", "Inventory API", "Customer API"],
    "mcp_server_description": ["...", "...", "..."],
})
result = flow.generate(df, runtime_params={"multiply_tool_rows": {"num_samples": 50}})
```

---
## Summary

This tutorial walked through the full **MCP Server Distillation** lifecycle:

1. **Demo MCP Server** — A 15-tool e-commerce analytics platform with deliberate ambiguity clusters
2. **Expert Exploration** — A frontier model actively called every tool, discovering real data entities and tool relationships
3. **Question Synthesis** — A teacher LLM generated realistic questions grounded in exploration findings
4. **Expert Trajectories** — The frontier model solved each question via actual MCP tool calls
5. **Training Data Formatting** — Tool traces converted into structured function-calling conversations
6. **GRPO Training** — LoRA fine-tuning with group relative policy optimization

### What makes this different

- **Active exploration, not passive schema reading** — The model discovers real data and tool behavior
- **Expert trajectories, not student filtering** — A strong model generates consistently good examples
- **Grounded questions** — Questions reference real entities and exploit known tool relationships
- **Works with any MCP server** — Just point the pipeline at your Langflow agent + MCP server

### Next steps

- Try with your own MCP server and tools
- Scale up to 50-100 examples with higher tool-per-question counts
- Use the CLI scripts in `examples/` for production pipelines
- Deploy the trained model with vLLM for production inference